# 1. Load the 000_dataset_v2 Split Metadata

In [1]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "000_dataset_v2").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

METADATA_DIR = PROJECT_ROOT / "000_dataset_v2" / "_segmentation_dataset"
HOLDOUT_CSV = METADATA_DIR / "001_holdout_split_lidc_lndb.csv"
CV_CSV = METADATA_DIR / "004_classification_cv_5fold_seed42.csv"

holdout = pd.read_csv(HOLDOUT_CSV)
cv = pd.read_csv(CV_CSV)

filename_pattern = (
    r"^(?P<study_id>.+)_"
    r"(?P<nodule_token>(?:cluster|finding)_\d+)_"
    r"slice_(?P<slice_number>\d+)\.npy$"
)


def add_identifiers(data):
    """Add CT scan, nodule, and slice identifiers."""
    identifiers = data["filename"].str.extract(filename_pattern)
    result = data.copy()
    result["study_id"] = identifiers["study_id"]
    result["nodule_id"] = (
        result["dataset"]
        + "::"
        + identifiers["study_id"]
        + "::"
        + identifiers["nodule_token"]
    )
    result["slice_number"] = identifiers["slice_number"].astype(int)
    result["slice_id"] = (
        result["dataset"]
        + "::"
        + identifiers["study_id"]
        + "::slice_"
        + identifiers["slice_number"]
    )
    result["label"] = result["label"].str.lower()
    return result


holdout = add_identifiers(holdout)
cv = add_identifiers(cv)

dataset_scopes = (
    ("LIDC-IDRI + LNDb", None),
    ("LIDC-IDRI", "LIDC-IDRI"),
    ("LNDb", "LNDb"),
)
partition_colors = {
    "Train": "#4C78A8",
    "Validation": "#F2CF5B",
    "Test": "#E45756",
}
scope_colors = {
    "LIDC-IDRI + LNDb": "#4C78A8",
    "LIDC-IDRI": "#72B7B2",
    "LNDb": "#E45756",
}

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 8,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
})


def select_dataset(data, dataset_name):
    """Select the combined dataset or one source dataset."""
    if dataset_name is None:
        return data
    return data.loc[data["dataset"].eq(dataset_name)]


def unique_nodules(data):
    """Return one row for each nodule."""
    return data.drop_duplicates("nodule_id")


def format_axis(axis):
    """Apply the shared compact plot style."""
    axis.grid(
        axis="y",
        color="#D9D9D9",
        linewidth=0.4,
        alpha=0.7,
    )
    axis.grid(axis="x", visible=False)
    axis.set_axisbelow(True)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)


sample_percent_format = {
    "Train (%)": "{:.2f}%",
    "Validation (%)": "{:.2f}%",
    "Test (%)": "{:.2f}%",
}
detail_percent_format = {
    "CT Scans (%)": "{:.2f}%",
    "Benign Nodules (%)": "{:.2f}%",
    "Malignant Nodules (%)": "{:.2f}%",
    "Total Nodules (%)": "{:.2f}%",
}

# 2. Overall Selected CT Scan and Nodule Distribution

In [2]:
overall_records = []

for scope_name, dataset_name in dataset_scopes:
    scoped_data = select_dataset(holdout, dataset_name)
    nodules = unique_nodules(scoped_data)

    overall_records.append({
        "Dataset Scope": scope_name,
        "Selected CT Scans": scoped_data["study_id"].nunique(),
        "Benign Nodules": nodules["label"].eq("benign").sum(),
        "Malignant Nodules": nodules["label"].eq("malignant").sum(),
        "Total Nodules": len(nodules),
    })

overall_distribution = pd.DataFrame(overall_records)
display(overall_distribution.style.format(thousands=","))

,Dataset Scope,Selected CT Scans,Benign Nodules,Malignant Nodules,Total Nodules
0,LIDC-IDRI + LNDb,709,918,470,"1,388"
1,LIDC-IDRI,508,420,382,802
2,LNDb,201,498,88,586


# 3. Patient-Level Holdout Distribution for U-Net Nodule Segmentation

## 3.1 Sample Distribution

In [3]:
holdout_splits = (
    ("Train", "train"),
    ("Validation", "val"),
    ("Test", "test"),
)
holdout_sample_records = []

for scope_name, dataset_name in dataset_scopes:
    scoped_data = select_dataset(holdout, dataset_name)
    total_samples = len(scoped_data)
    record = {"Dataset Scope": scope_name}

    for split_name, split_value in holdout_splits:
        sample_count = scoped_data["split"].eq(split_value).sum()
        record[f"{split_name} Samples"] = sample_count
        record[f"{split_name} (%)"] = 100 * sample_count / total_samples

    record["Total Samples"] = total_samples
    holdout_sample_records.append(record)

holdout_sample_distribution = pd.DataFrame(holdout_sample_records)
display(
    holdout_sample_distribution.style.format(
        sample_percent_format,
        thousands=",",
    )
)

,Dataset Scope,Train Samples,Train (%),Validation Samples,Validation (%),Test Samples,Test (%),Total Samples
0,LIDC-IDRI + LNDb,"5,856",69.56%,"1,253",14.88%,"1,310",15.56%,"8,419"
1,LIDC-IDRI,"3,869",68.34%,925,16.34%,867,15.32%,"5,661"
2,LNDb,"1,987",72.04%,328,11.89%,443,16.06%,"2,758"


## 3.2 Selected CT Scan and Nodule Distribution

In [4]:
holdout_detail_records = []

for scope_name, dataset_name in dataset_scopes:
    scoped_data = select_dataset(holdout, dataset_name)
    scoped_nodules = unique_nodules(scoped_data)

    total_scans = scoped_data["study_id"].nunique()
    total_benign = scoped_nodules["label"].eq("benign").sum()
    total_malignant = scoped_nodules["label"].eq("malignant").sum()
    total_nodules = len(scoped_nodules)

    for split_name, split_value in holdout_splits:
        partition = scoped_data.loc[
            scoped_data["split"].eq(split_value)
        ]
        partition_nodules = unique_nodules(partition)

        ct_scans = partition["study_id"].nunique()
        benign = partition_nodules["label"].eq("benign").sum()
        malignant = partition_nodules["label"].eq("malignant").sum()
        nodule_count = len(partition_nodules)

        holdout_detail_records.append({
            "Dataset Scope": scope_name,
            "Split": split_name,
            "Selected CT Scans": ct_scans,
            "CT Scans (%)": 100 * ct_scans / total_scans,
            "Benign Nodules": benign,
            "Benign Nodules (%)": 100 * benign / total_benign,
            "Malignant Nodules": malignant,
            "Malignant Nodules (%)": 100 * malignant / total_malignant,
            "Total Nodules": nodule_count,
            "Total Nodules (%)": 100 * nodule_count / total_nodules,
        })

holdout_detail_distribution = pd.DataFrame(holdout_detail_records)
display(
    holdout_detail_distribution.style.format(
        detail_percent_format,
        thousands=",",
    )
)

,Dataset Scope,Split,Selected CT Scans,CT Scans (%),Benign Nodules,Benign Nodules (%),Malignant Nodules,Malignant Nodules (%),Total Nodules,Total Nodules (%)
0,LIDC-IDRI + LNDb,Train,495,69.82%,667,72.66%,316,67.23%,983,70.82%
1,LIDC-IDRI + LNDb,Validation,106,14.95%,115,12.53%,87,18.51%,202,14.55%
2,LIDC-IDRI + LNDb,Test,108,15.23%,136,14.81%,67,14.26%,203,14.63%
3,LIDC-IDRI,Train,355,69.88%,312,74.29%,256,67.02%,568,70.82%
4,LIDC-IDRI,Validation,76,14.96%,51,12.14%,73,19.11%,124,15.46%
5,LIDC-IDRI,Test,77,15.16%,57,13.57%,53,13.87%,110,13.72%
6,LNDb,Train,140,69.65%,355,71.29%,60,68.18%,415,70.82%
7,LNDb,Validation,30,14.93%,64,12.85%,14,15.91%,78,13.31%
8,LNDb,Test,31,15.42%,79,15.86%,14,15.91%,93,15.87%


# 4. Five-Fold Cross-Validation Distribution for Baseline ResNet-50 and Multi-Stage Guided ResNet-50

## 4.1 Development and Test Distribution

In [5]:
development_test_splits = (
    ("Development", "development"),
    ("Test", "holdout_test"),
)
development_test_records = []

for scope_name, dataset_name in dataset_scopes:
    scoped_data = select_dataset(cv, dataset_name)
    scoped_nodules = unique_nodules(scoped_data)

    total_scans = scoped_data["study_id"].nunique()
    total_benign = scoped_nodules["label"].eq("benign").sum()
    total_malignant = scoped_nodules["label"].eq("malignant").sum()
    total_nodules = len(scoped_nodules)

    for split_name, split_value in development_test_splits:
        split_data = scoped_data.loc[
            scoped_data["cv_role"].eq(split_value)
        ]
        split_nodules = unique_nodules(split_data)

        selected_scans = split_data["study_id"].nunique()
        benign_nodules = split_nodules["label"].eq("benign").sum()
        malignant_nodules = (
            split_nodules["label"].eq("malignant").sum()
        )
        nodule_count = len(split_nodules)

        development_test_records.append({
            "Dataset Scope": scope_name,
            "Split": split_name,
            "Selected CT Scans": selected_scans,
            "CT Scans %": 100 * selected_scans / total_scans,
            "Benign Nodules": benign_nodules,
            "Benign Nodules (%)": 100 * benign_nodules / total_benign,
            "Malignant Nodules": malignant_nodules,
            "Malignant Nodules (%)": (
                100 * malignant_nodules / total_malignant
            ),
            "Total Nodules": nodule_count,
            "Total Nodules(%)": 100 * nodule_count / total_nodules,
        })

development_test_distribution = pd.DataFrame(
    development_test_records
)
development_test_percent_format = {
    "CT Scans %": "{:.2f}%",
    "Benign Nodules (%)": "{:.2f}%",
    "Malignant Nodules (%)": "{:.2f}%",
    "Total Nodules(%)": "{:.2f}%",
}
display(
    development_test_distribution.style.format(
        development_test_percent_format,
        thousands=",",
    )
)

,Dataset Scope,Split,Selected CT Scans,CT Scans %,Benign Nodules,Benign Nodules (%),Malignant Nodules,Malignant Nodules (%),Total Nodules,Total Nodules(%)
0,LIDC-IDRI + LNDb,Development,601,84.77%,782,85.19%,403,85.74%,"1,185",85.37%
1,LIDC-IDRI + LNDb,Test,108,15.23%,136,14.81%,67,14.26%,203,14.63%
2,LIDC-IDRI,Development,431,84.84%,363,86.43%,329,86.13%,692,86.28%
3,LIDC-IDRI,Test,77,15.16%,57,13.57%,53,13.87%,110,13.72%
4,LNDb,Development,170,84.58%,419,84.14%,74,84.09%,493,84.13%
5,LNDb,Test,31,15.42%,79,15.86%,14,15.91%,93,15.87%


## 4.2 Sample Distribution

In [6]:
def add_cv_partition(data, validation_fold):
    """Add Train, Validation, and Test labels for one fold."""
    result = data.copy()
    result["partition"] = "Train"
    result.loc[
        result["cv_role"].eq("holdout_test"),
        "partition",
    ] = "Test"
    result.loc[
        result["cv_role"].eq("development")
        & result["cv_fold"].eq(validation_fold),
        "partition",
    ] = "Validation"
    return result


partition_order = ("Train", "Validation", "Test")
cv_sample_records = []

for fold in range(5):
    fold_data = add_cv_partition(cv, fold)

    for scope_name, dataset_name in dataset_scopes:
        scoped_data = select_dataset(fold_data, dataset_name)
        total_samples = len(scoped_data)
        record = {
            "Validation Fold": fold,
            "Dataset Scope": scope_name,
        }

        for partition_name in partition_order:
            sample_count = scoped_data["partition"].eq(
                partition_name
            ).sum()
            record[f"{partition_name} Samples"] = sample_count
            record[f"{partition_name} (%)"] = (
                100 * sample_count / total_samples
            )

        record["Total Samples"] = total_samples
        cv_sample_records.append(record)

cv_sample_distribution = pd.DataFrame(cv_sample_records)
display(
    cv_sample_distribution.style.format(
        sample_percent_format,
        thousands=",",
    )
)

,Validation Fold,Dataset Scope,Train Samples,Train (%),Validation Samples,Validation (%),Test Samples,Test (%),Total Samples
0,0,LIDC-IDRI + LNDb,"5,527",65.65%,"1,582",18.79%,"1,310",15.56%,"8,419"
1,0,LIDC-IDRI,"3,751",66.26%,"1,043",18.42%,867,15.32%,"5,661"
2,0,LNDb,"1,776",64.39%,539,19.54%,443,16.06%,"2,758"
3,1,LIDC-IDRI + LNDb,"5,692",67.61%,"1,417",16.83%,"1,310",15.56%,"8,419"
4,1,LIDC-IDRI,"3,889",68.70%,905,15.99%,867,15.32%,"5,661"
5,1,LNDb,"1,803",65.37%,512,18.56%,443,16.06%,"2,758"
6,2,LIDC-IDRI + LNDb,"5,797",68.86%,"1,312",15.58%,"1,310",15.56%,"8,419"
7,2,LIDC-IDRI,"3,841",67.85%,953,16.83%,867,15.32%,"5,661"
8,2,LNDb,"1,956",70.92%,359,13.02%,443,16.06%,"2,758"
9,3,LIDC-IDRI + LNDb,"5,805",68.95%,"1,304",15.49%,"1,310",15.56%,"8,419"


## 4.3 Selected CT Scan and Nodule Distribution

In [7]:
cv_detail_records = []

for fold in range(5):
    fold_data = add_cv_partition(cv, fold)

    for scope_name, dataset_name in dataset_scopes:
        scoped_data = select_dataset(fold_data, dataset_name)
        scoped_nodules = unique_nodules(scoped_data)

        total_scans = scoped_data["study_id"].nunique()
        total_benign = scoped_nodules["label"].eq("benign").sum()
        total_malignant = scoped_nodules["label"].eq("malignant").sum()
        total_nodules = len(scoped_nodules)

        for partition_name in partition_order:
            partition = scoped_data.loc[
                scoped_data["partition"].eq(partition_name)
            ]
            partition_nodules = unique_nodules(partition)

            ct_scans = partition["study_id"].nunique()
            benign = partition_nodules["label"].eq("benign").sum()
            malignant = partition_nodules["label"].eq("malignant").sum()
            nodule_count = len(partition_nodules)

            cv_detail_records.append({
                "Validation Fold": fold,
                "Dataset Scope": scope_name,
                "Partition": partition_name,
                "Selected CT Scans": ct_scans,
                "CT Scans (%)": 100 * ct_scans / total_scans,
                "Benign Nodules": benign,
                "Benign Nodules (%)": 100 * benign / total_benign,
                "Malignant Nodules": malignant,
                "Malignant Nodules (%)": 100 * malignant / total_malignant,
                "Total Nodules": nodule_count,
                "Total Nodules (%)": 100 * nodule_count / total_nodules,
            })

cv_detail_distribution = pd.DataFrame(cv_detail_records)
display(
    cv_detail_distribution.style.format(
        detail_percent_format,
        thousands=",",
    )
)

,Validation Fold,Dataset Scope,Partition,Selected CT Scans,CT Scans (%),Benign Nodules,Benign Nodules (%),Malignant Nodules,Malignant Nodules (%),Total Nodules,Total Nodules (%)
0,0,LIDC-IDRI + LNDb,Train,481,67.84%,612,66.67%,330,70.21%,942,67.87%
1,0,LIDC-IDRI + LNDb,Validation,120,16.93%,170,18.52%,73,15.53%,243,17.51%
2,0,LIDC-IDRI + LNDb,Test,108,15.23%,136,14.81%,67,14.26%,203,14.63%
3,0,LIDC-IDRI,Train,342,67.32%,275,65.48%,274,71.73%,549,68.45%
4,0,LIDC-IDRI,Validation,89,17.52%,88,20.95%,55,14.40%,143,17.83%
5,0,LIDC-IDRI,Test,77,15.16%,57,13.57%,53,13.87%,110,13.72%
6,0,LNDb,Train,139,69.15%,337,67.67%,56,63.64%,393,67.06%
7,0,LNDb,Validation,31,15.42%,82,16.47%,18,20.45%,100,17.06%
8,0,LNDb,Test,31,15.42%,79,15.86%,14,15.91%,93,15.87%
9,1,LIDC-IDRI + LNDb,Train,480,67.70%,597,65.03%,322,68.51%,919,66.21%


# 5. Data Leakage Validation at CT Scan, Slice, and Nodule Levels

## 5.1 Patient-Level Holdout Leakage Check

In [8]:
partition_pairs = (
    ("Train", "Validation"),
    ("Train", "Test"),
    ("Validation", "Test"),
)


def count_overlap(left, right, column):
    """Count shared identifiers between two partitions."""
    left_ids = set(left[column])
    right_ids = set(right[column])
    return len(left_ids & right_ids)


holdout_leakage_records = []

for scope_name, dataset_name in dataset_scopes:
    scoped_data = select_dataset(holdout, dataset_name).copy()
    scoped_data["partition"] = scoped_data["split"].replace({
        "train": "Train",
        "val": "Validation",
        "test": "Test",
    })

    for left_name, right_name in partition_pairs:
        left = scoped_data.loc[
            scoped_data["partition"].eq(left_name)
        ]
        right = scoped_data.loc[
            scoped_data["partition"].eq(right_name)
        ]

        scan_overlap = count_overlap(left, right, "study_id")
        slice_overlap = count_overlap(left, right, "slice_id")
        nodule_overlap = count_overlap(left, right, "nodule_id")

        holdout_leakage_records.append({
            "Dataset Scope": scope_name,
            "Split Pair": f"{left_name} vs {right_name}",
            "CT Scan Overlap": scan_overlap,
            "Slice Overlap": slice_overlap,
            "Nodule Overlap": nodule_overlap,
            "Status": (
                "No Leakage"
                if scan_overlap + slice_overlap + nodule_overlap == 0
                else "Leakage Detected"
            ),
        })

holdout_leakage = pd.DataFrame(holdout_leakage_records)
display(holdout_leakage)

,Dataset Scope,Split Pair,CT Scan Overlap,Slice Overlap,Nodule Overlap,Status
0,LIDC-IDRI + LNDb,Train vs Validation,0,0,0,No Leakage
1,LIDC-IDRI + LNDb,Train vs Test,0,0,0,No Leakage
2,LIDC-IDRI + LNDb,Validation vs Test,0,0,0,No Leakage
3,LIDC-IDRI,Train vs Validation,0,0,0,No Leakage
4,LIDC-IDRI,Train vs Test,0,0,0,No Leakage
5,LIDC-IDRI,Validation vs Test,0,0,0,No Leakage
6,LNDb,Train vs Validation,0,0,0,No Leakage
7,LNDb,Train vs Test,0,0,0,No Leakage
8,LNDb,Validation vs Test,0,0,0,No Leakage


## 5.2 Five-Fold Cross-Validation Leakage Check

In [9]:
cv_leakage_records = []

for fold in range(5):
    fold_data = add_cv_partition(cv, fold)

    for scope_name, dataset_name in dataset_scopes:
        scoped_data = select_dataset(fold_data, dataset_name)

        for left_name, right_name in partition_pairs:
            left = scoped_data.loc[
                scoped_data["partition"].eq(left_name)
            ]
            right = scoped_data.loc[
                scoped_data["partition"].eq(right_name)
            ]

            scan_overlap = count_overlap(left, right, "study_id")
            slice_overlap = count_overlap(left, right, "slice_id")
            nodule_overlap = count_overlap(left, right, "nodule_id")

            cv_leakage_records.append({
                "Validation Fold": fold,
                "Dataset Scope": scope_name,
                "Split Pair": f"{left_name} vs {right_name}",
                "CT Scan Overlap": scan_overlap,
                "Slice Overlap": slice_overlap,
                "Nodule Overlap": nodule_overlap,
                "Status": (
                    "No Leakage"
                    if scan_overlap + slice_overlap + nodule_overlap == 0
                    else "Leakage Detected"
                ),
            })

cv_leakage = pd.DataFrame(cv_leakage_records)
display(cv_leakage)

,Validation Fold,Dataset Scope,Split Pair,CT Scan Overlap,Slice Overlap,Nodule Overlap,Status
0,0,LIDC-IDRI + LNDb,Train vs Validation,0,0,0,No Leakage
1,0,LIDC-IDRI + LNDb,Train vs Test,0,0,0,No Leakage
2,0,LIDC-IDRI + LNDb,Validation vs Test,0,0,0,No Leakage
3,0,LIDC-IDRI,Train vs Validation,0,0,0,No Leakage
4,0,LIDC-IDRI,Train vs Test,0,0,0,No Leakage
5,0,LIDC-IDRI,Validation vs Test,0,0,0,No Leakage
6,0,LNDb,Train vs Validation,0,0,0,No Leakage
7,0,LNDb,Train vs Test,0,0,0,No Leakage
8,0,LNDb,Validation vs Test,0,0,0,No Leakage
9,1,LIDC-IDRI + LNDb,Train vs Validation,0,0,0,No Leakage
